<a href="https://colab.research.google.com/github/filippopedrini95/MS_DF_ComputerVision_Project/blob/main/ComputerVisionProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Libraries

In [1]:
from pathlib import Path
import time
import pandas as pd
import numpy as np
import keras
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.datasets import cifar10
from sklearn.metrics import classification_report
from colorama import Fore, Style

I0000 00:00:1773317662.778344   20719 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773317662.778934   20719 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1773317662.834028   20719 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773317664.223785   20719 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

#Import Dataset & Images Preprocessing

In [2]:
#import image dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

/workspaces/MS_DF_ComputerVision_Project/.venv/lib/python3.12/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


In [3]:
x_test.shape

(10000, 32, 32, 3)

## EfficientNetB0

### image preprocessing for EfficientNetB0

test dataset

In [4]:
#image preprocessing to match ResNet input requirements
x_test_effnetb0 = efficientnet_preprocess(x_test)

full dataset

In [5]:
#image preprocessing to match ResNet input requirements
x_train_effnetb0 = efficientnet_preprocess(x_train)

In [6]:
#validation set
x_train_effnetb0, x_val, y_train_effnetb0, y_val = train_test_split(
    x_train_effnetb0,
    y_train,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y_train
)

### No Feature Extractor Optim

In [7]:
#import EfficientNetB0 for feature extraction
pretrained_model = keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(32,32,3),
    pooling="max"
)

pretrained_model.trainable = False # disable EffocoemtNetB0 parameter optimization

#define the structure of the CNN
cnn_model = Sequential([
    pretrained_model,
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(10, activation='softmax')
])

#compile the model
cnn_model.compile(optimizer="adam",
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

#define early stopping
early_stop = EarlyStopping(
    monitor="val_loss",         # monitors validation loss function
    patience=4,                 # number of steps with no improvements in "val_loss" before stopping the fitting
    restore_best_weights=True   # in case the further "patience" steps made the performance worse, it restores the best weights
)

E0000 00:00:1773317668.996035   20719 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [8]:
start_time = time.time()

#model fitting
history = cnn_model.fit(
    x=x_train_effnetb0,
    y=y_train_effnetb0,
    epochs= 1, #50,
    batch_size=64, #default: 32
    validation_data=(x_val, y_val),
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)

625/625 ━━━━━━━━━━━━━━━━━━━━ 62s 88ms/step - accuracy: 0.4167 - loss: 1.6103 - val_accuracy: 0.5182 - val_loss: 1.3359


In [9]:
end_time = time.time()
print(Fore.GREEN + f"\ntraining completed - duration:{(end_time - start_time)/60:.1f} min" + Style.RESET_ALL)


training completed - duration:1.0 min


In [10]:
#save the model after training
#cnn_model.save("effnetb0_dense128_dropout02_dense32.keras")

In [11]:
#model prediction
y_pred_probs = cnn_model.predict(x_test_effnetb0)
y_pred = np.argmax(y_pred_probs, axis=1)

#model evaluation parameters
test_loss, test_accuracy = cnn_model.evaluate(x_test_effnetb0, y_test)
print("\naccuracy:", test_accuracy)
print("loss:", test_loss, end="\n\n")

313/313 ━━━━━━━━━━━━━━━━━━━━ 15s 44ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 41ms/step - accuracy: 0.5167 - loss: 1.3477

accuracy: 0.516700029373169
loss: 1.3477075099945068



In [12]:
#classification report
class_names = [
        "airplane", "automobile", "bird", "cat", "deer",
        "dog", "frog", "horse", "ship", "truck"
]

print(classification_report(y_test, y_pred, target_names=class_names))

              precision    recall  f1-score   support

    airplane       0.63      0.52      0.57      1000
  automobile       0.57      0.55      0.56      1000
        bird       0.52      0.35      0.42      1000
         cat       0.36      0.32      0.33      1000
        deer       0.55      0.42      0.48      1000
         dog       0.38      0.54      0.45      1000
        frog       0.65      0.61      0.63      1000
       horse       0.50      0.66      0.57      1000
        ship       0.62      0.53      0.57      1000
       truck       0.50      0.68      0.57      1000

    accuracy                           0.52     10000
   macro avg       0.53      0.52      0.51     10000
weighted avg       0.53      0.52      0.51     10000

